# Semantic Segmentation

Last week we classified a **whole image** into one label. **Semantic segmentation** goes finer: we classify **every pixel**. A segmentation network maps an image of shape `3 x H x W` to a tensor of per-pixel class scores `C x H x W` (`C` = number of classes); taking the `argmax` over the class dimension gives the predicted label map. So segmentation is simply **per-pixel classification**.

In this tutorial we will:
1. Write our own `Dataset` classes for two segmentation datasets (**CamVid** and **DeepCrack**).
2. Build a **UNet** decoder on top of a pretrained ResNet18 encoder (**Task 1**).
3. Implement the **Dice loss** for a strongly imbalanced binary problem (**Task 2**).
4. Implement an **atrous (dilated) ASPP module** and measure what it buys us (**Task 3**).

Let's load in any libraries we will use in this notebook.

In [ ]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

#import torch which has many of the functions to build deep learning models and to train them
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset

#import torchvision for loading and working with image data
import torchvision.transforms as T
import torchvision.transforms.functional as TF
from torchvision.models import resnet18, ResNet18_Weights

#this is a nice progress bar representation that will be good to measure progress during training
import tqdm

# fix seed for reproducibility
torch.manual_seed(0)
np.random.seed(0)
random.seed(0)

# setup device
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu') #this line checks if we have a GPU available
print(f"Using device: {device}")

# 1. The Data

We start with [**CamVid**](http://mi.eng.cam.ac.uk/research/projects/VideoRec/CamVid/), a street-scene dataset where every pixel of every frame is labelled.

[`torchvision.datasets.ImageFolder`](https://pytorch.org/vision/stable/generated/torchvision.datasets.ImageFolder.html) only supports single-label classification, so for segmentation we have to write **our own dataset class**, which returns an image **and** its corresponding segmentation mask. Let's first open one sample to see what the data looks like.

In [ ]:
camvid_root = "data/camvid"

first_image = f'{camvid_root}/rgb/0001TP_006690.jpg'
first_label = f'{camvid_root}/labels_gray/0001TP_006690_L.png'

img = Image.open(first_image)
lbl = Image.open(first_label)

img_array = np.array(img)
lbl_array = np.array(lbl)

print(img_array.shape, lbl_array.shape)
print(np.unique(lbl_array))

The mask is a single-channel image: each pixel holds a **class index**, not a colour. Here the indices run from 0 to 10, plus **255**, which is the *void* class (unlabelled or ambiguous pixels). To display the mask we map each index to an arbitrary colour.

In [ ]:
# we have labels 0-10 plus 255, we need to remap these for rgb display
color_map = {0: (0, 0, 0), 1: (128, 0, 0), 2: (0, 128, 0), 3: (128, 128, 0), 4: (0, 0, 128),
             5: (128, 0, 128), 6: (0, 128, 128), 8: (128, 128, 128), 9: (64, 0, 0), 255: (255, 255, 255)}

lbl_rgb = np.zeros((lbl_array.shape[0], lbl_array.shape[1], 3), dtype=np.uint8)
for label_val, rgb in color_map.items():
    lbl_rgb[lbl_array == label_val] = rgb # remap with our colors

plt.figure(figsize=(15, 5))
plt.subplot(1, 3, 1)
plt.imshow(img_array)
plt.title("Image")
plt.axis('off')
plt.subplot(1, 3, 2)
plt.imshow(lbl_rgb)
plt.title("Label")
plt.axis('off')
plt.subplot(1, 3, 3)
plt.imshow(img_array)
plt.imshow(lbl_rgb, alpha=0.5)
plt.title("Overlay")
plt.axis('off')
plt.show()

For this tutorial, we are going to focus on a simpler problem, with only a few selected classes: **Road**, **Car**, **Pedestrian**, and **Other**. The remaining classes are merged into *Other*, and the void class is kept as 255 so it can be **ignored** by the loss.

In [ ]:
convertor = {
    0: 3,    # Sky        => other
    1: 3,    # Building   => other
    2: 3,    # Pole       => other
    3: 0,    # Road       keep
    4: 3,    # Sidewalk   => other
    5: 3,    # Tree       => other
    6: 3,    # SignSymbol => other
    7: 3,    # Fence      => other
    8: 1,    # Car        keep
    9: 2,    # Pedestrian keep
    10: 3,   # Bicyclist  => other
    255: 255 # Void       => ignore (unlabelled / ambiguous class)
}

simple_lbl_array = np.copy(lbl_array)
for original_label, new_label in convertor.items():
    simple_lbl_array[lbl_array == original_label] = new_label

simple_lbl_rgb = np.zeros((simple_lbl_array.shape[0], simple_lbl_array.shape[1], 3), dtype=np.uint8)
for label_val, rgb in color_map.items():
    simple_lbl_rgb[simple_lbl_array == label_val] = rgb

plt.figure(figsize=(15, 5))
plt.subplot(1, 3, 1)
plt.imshow(img_array)
plt.title("Image")
plt.axis('off')
plt.subplot(1, 3, 2)
plt.imshow(simple_lbl_rgb)
plt.title("Simplified Label")
plt.axis('off')
plt.subplot(1, 3, 3)
plt.imshow(img_array)
plt.imshow(simple_lbl_rgb, alpha=0.5)
plt.title("Overlay")
plt.axis('off')
plt.show()

## The `CamVid` dataset class

A PyTorch [`Dataset`](https://pytorch.org/docs/stable/data.html#torch.utils.data.Dataset) needs two methods: `__len__` (how many samples) and `__getitem__` (return sample `idx`). CamVid ships **split files** (`train.txt`, `val.txt`, `test.txt`), one `image label` pair per line, so `__init__` only has to read that file and build the two lists of paths.

Two points specific to segmentation:
- the mask must be resized with [**nearest-neighbour**](https://pytorch.org/vision/stable/generated/torchvision.transforms.functional.resize.html) interpolation — bilinear would average class indices and invent labels that do not exist;
- the image is normalised with the ImageNet statistics (our encoder is pretrained), but the mask is **not** normalised: it is a `long` tensor of class indices.

In [ ]:
W_camvid, H_camvid = 384, 288 # resize to dimensions that are adequate for ResNet

normalise_camvid = T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

class CamVid(Dataset):
    def __init__(self, root_dir, split_file, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        with open(split_file, 'r') as f:
            lines = [line.strip() for line in f if line.strip()]

        self.image_paths = []
        self.label_paths = []
        for line in lines:
            img_rel, lab_rel = line.split()
            self.image_paths.append(f"{self.root_dir}/{img_rel}")
            self.label_paths.append(f"{self.root_dir}/{lab_rel}")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image_path = self.image_paths[idx]
        label_path = self.label_paths[idx]

        image = Image.open(image_path).convert('RGB')
        label = Image.open(label_path)

        # Resize (nearest for the mask: no interpolation between class indices)
        image = TF.resize(image, (H_camvid, W_camvid))
        label = TF.resize(label, (H_camvid, W_camvid), interpolation=Image.NEAREST)

        # Remap the 11 classes => 4 classes
        label_np = np.array(label, dtype=np.uint8)
        label_new = np.full_like(label_np, 3)          # default => other
        for original_label, new_label in convertor.items():
            label_new[label_np == original_label] = new_label

        # Convert to tensor
        image = TF.to_tensor(image)
        label = torch.from_numpy(label_new).long()

        # Normalize
        if self.transform:
            image = self.transform(image)

        return image, label

In [ ]:
train_dataset = CamVid(root_dir=camvid_root, split_file=f"{camvid_root}/train.txt", transform=normalise_camvid)
val_dataset   = CamVid(root_dir=camvid_root, split_file=f"{camvid_root}/val.txt",   transform=normalise_camvid)
test_dataset  = CamVid(root_dir=camvid_root, split_file=f"{camvid_root}/test.txt",  transform=normalise_camvid)

print(f"size of training dataset: {len(train_dataset)}")
print(f"size of validation dataset: {len(val_dataset)}")
print(f"size of test dataset: {len(test_dataset)}")

Always check one sample coming out of the dataset before training: the image should be a `3 x H x W` float tensor, the mask a `H x W` long tensor holding only the indices we expect (`0, 1, 2, 3` and possibly `255`).

In [ ]:
training_image, training_mask = train_dataset[1]

print(f"image tensor: {list(training_image.shape)}, mask tensor: {list(training_mask.shape)}")
print(f"labels present in this mask: {torch.unique(training_mask).tolist()}")

label_rgb = np.zeros((training_mask.shape[0], training_mask.shape[1], 3), dtype=np.uint8)
for label_val, rgb in color_map.items():
    label_rgb[training_mask.numpy() == label_val] = rgb

plt.imshow(training_image.permute(1, 2, 0).numpy())
plt.imshow(label_rgb, alpha=0.5)
plt.axis('off')
plt.show()

# 2. The Model: UNet

We are going to use a **UNet**, a symmetric encoder-decoder architecture with **skip connections**:

- the **encoder** progressively downsamples the image, producing feature maps at strides /4, /8, /16 and /32. Deep features carry *semantics* (what is in the image) but have lost spatial precision;
- the **decoder** upsamples back to full resolution;
- the **skip connections** concatenate each encoder feature map with the decoder feature map of the same stride, so the decoder recovers the *localisation* the encoder threw away.

As in the previous tutorial we do not train the encoder from scratch: we reuse a **ResNet18 pretrained on ImageNet** and freeze it, so only the decoder is learned.

The cell below provides the three building blocks you will need:
- `conv_bn_relu` — the standard conv + batch norm + ReLU triplet (note the `dilation` argument, we will come back to it in Task 3);
- `ResNet18Encoder` — the frozen backbone, returning the four feature maps `c4, c8, c16, c32`;
- `DecoderBlock` — upsamples by 2, concatenates the skip connection, and applies two convolutions.

In [ ]:
def conv_bn_relu(cin, cout, dilation=1):
    return nn.Sequential(
        nn.Conv2d(cin, cout, 3, padding=dilation, dilation=dilation, bias=False),
        nn.BatchNorm2d(cout),
        nn.ReLU(inplace=True),)


class ResNet18Encoder(nn.Module):
    def __init__(self, freeze=True):
        super().__init__()
        net = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
        self.stem    = nn.Sequential(net.conv1, net.bn1, net.relu)  # /2,   64
        self.maxpool = net.maxpool
        self.layer1  = net.layer1   # /4,   64
        self.layer2  = net.layer2   # /8,  128
        self.layer3  = net.layer3   # /16, 256
        self.layer4  = net.layer4   # /32, 512
        if freeze:
            for p in self.parameters():
                p.requires_grad_(False)

    def forward(self, x):
        x   = self.stem(x)
        x   = self.maxpool(x)
        c4  = self.layer1(x)    # stride /4
        c8  = self.layer2(c4)   # stride /8
        c16 = self.layer3(c8)   # stride /16
        c32 = self.layer4(c16)  # stride /32
        return c4, c8, c16, c32


class DecoderBlock(nn.Module):
    def __init__(self, cin, cskip, cout):
        super().__init__()
        self.conv1 = conv_bn_relu(cin + cskip, cout)
        self.conv2 = conv_bn_relu(cout, cout)

    def forward(self, x, skip):
        x = F.interpolate(x, scale_factor=2, mode="bilinear", align_corners=False) # upsample by 2x
        x = torch.cat([x, skip], dim=1)                                            # skip connection
        return self.conv2(self.conv1(x))

Before writing the decoder, let's look at the **shapes** the encoder produces: they tell us how many channels each `DecoderBlock` has to consume.

In [ ]:
enc = ResNet18Encoder()
c4, c8, c16, c32 = enc(training_image.unsqueeze(0)) # we need to add an extra dimension for the batch

print(f'Size of the encoder layers c4: {list(c4.shape)}')
print(f'Size of the encoder layers c8: {list(c8.shape)}')
print(f'Size of the encoder layers c16: {list(c16.shape)}')
print(f'Size of the encoder layers c32: {list(c32.shape)}')

### Task 1: implement the UNet decoder

Complete the `UNetLite` model below using the `DecoderBlock` provided above.

1. In `__init__`, create the **three decoder blocks**. Each block takes `cin` (channels coming from the previous stage), `cskip` (channels of the encoder feature map it is fused with) and `cout` (channels it produces):
   - `dec3`: from `c32` (512 channels), skip `c16` (256), output 256;
   - `dec2`: from the previous output (256), skip `c8` (128), output 128;
   - `dec1`: from the previous output (128), skip `c4` (64), output 64.
2. Create the **prediction head**: a [`nn.Conv2d`](https://pytorch.org/docs/stable/generated/torch.nn.Conv2d.html) with `kernel_size=1` mapping the 64 decoder channels to `num_classes` **logits** (one score per class, per pixel).
3. In `forward`, chain the three decoder blocks, each one taking the previous output and the **matching skip connection**, then apply the head.

**Hint:** the last decoder output is at stride /4, so the final [`F.interpolate`](https://pytorch.org/docs/stable/generated/torch.nn.functional.interpolate.html) (already written) brings the logits back to the input resolution `H x W`. No softmax here: [`nn.CrossEntropyLoss`](https://pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html) expects raw logits.

In [ ]:
class UNetLite(nn.Module):
    def __init__(self, num_classes, freeze_encoder=True):
        super().__init__()
        self.encoder = ResNet18Encoder(freeze=freeze_encoder)

        # decoder: three blocks, each one fusing the previous output with an encoder skip
        self.dec3 = ...
        self.dec2 = ...
        self.dec1 = ...

        # 1x1 conv mapping the decoder features to per-pixel class logits
        self.head = ...

    def forward(self, x):
        H, W = x.shape[-2:]
        c4, c8, c16, c32 = self.encoder(x)

        # TODO: run the three decoder blocks then the head, x should end up as the logits
        x = ...

        # back to the input resolution
        return F.interpolate(x, size=(H, W), mode="bilinear", align_corners=False)

A quick shape check: for a batch of `B` images of size `H x W`, the model must output `B x num_classes x H x W`.

In [ ]:
model = UNetLite(num_classes=4, freeze_encoder=True)
logits = model(training_image.unsqueeze(0))
print(f"input: {list(training_image.unsqueeze(0).shape)} => logits: {list(logits.shape)}")
print(f"predicted label map: {list(logits.argmax(1).shape)}")

# 3. Training the Model

The training loop is the same as last week, with two segmentation-specific details:

1. **The loss.** `nn.CrossEntropyLoss` accepts a `C x H x W` logit map against a `H x W` map of class indices, and averages over all pixels: segmentation really is per-pixel classification. We pass `ignore_index=255` so that **void pixels do not contribute** to the loss.
2. **The metric.** Pixel accuracy is a poor metric here — most pixels are road or *other*, so a model predicting only those already scores well. We instead use the **Intersection over Union (IoU)**, computed per class and then averaged (**mIoU**):

$$\text{IoU}_c = \frac{TP_c}{TP_c + FP_c + FN_c}, \qquad \text{mIoU} = \frac{1}{C}\sum_c \text{IoU}_c.$$

In [ ]:
def mean_iou(preds, masks, num_classes=4, ignore_index=255):
    """Macro-averaged IoU over valid classes."""
    ious = []
    for c in range(num_classes):
        valid = masks != ignore_index
        tp = ((preds == c) & (masks == c) & valid).sum().item()
        fp = ((preds == c) & (masks != c) & valid).sum().item()
        fn = ((preds != c) & (masks == c) & valid).sum().item()
        if tp + fp + fn > 0:
            ious.append(tp / (tp + fp + fn))
    return sum(ious) / len(ious) if ious else 0.0


def train_epoch(model, dataloader, criterion, optimiser, epoch, device):

    # Put the model in "train" mode
    model.train()

    train_loss = 0.0
    for imgs, masks in tqdm.tqdm(dataloader, total=len(dataloader), desc=f'Epoch {epoch+1} - training phase'):
        # get the images and masks from the dataloader and move to device (GPU or CPU)
        imgs, masks = imgs.to(device), masks.to(device)

        optimiser.zero_grad(set_to_none=True)
        loss = criterion(model(imgs), masks)
        loss.backward()
        optimiser.step()
        train_loss += loss.item()

    return train_loss / len(dataloader)


def eval_epoch(model, dataloader, criterion, epoch, device):

    # Put the model in "eval" mode
    model.eval()

    val_loss, all_preds, all_masks = 0.0, [], []
    with torch.no_grad(): # no computation graph is built, so no gradients are computed or stored
        for imgs, masks in tqdm.tqdm(dataloader, total=len(dataloader), desc=f'Epoch {epoch+1} - validation phase'):
            imgs, masks = imgs.to(device), masks.to(device)
            logits = model(imgs)
            val_loss += criterion(logits, masks).item()
            all_preds.append(logits.argmax(1).cpu())   # argmax over the class dimension
            all_masks.append(masks.cpu())

    miou = mean_iou(torch.cat(all_preds), torch.cat(all_masks))
    return val_loss / len(dataloader), miou

Now let's put everything together and train the model.

In [ ]:
# any hyperparameters
EPOCHS      = 10
LR          = 1e-4
BATCH       = 6
NUM_CLASSES = 4

train_loader = DataLoader(train_dataset, batch_size=BATCH, shuffle=True,  num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH, shuffle=False, num_workers=0, pin_memory=True)

# Initialise the model
model = UNetLite(num_classes=NUM_CLASSES, freeze_encoder=True)
model = model.to(device)

# Define a loss function
criterion = nn.CrossEntropyLoss(ignore_index=255)   # void pixels do not contribute to loss

# Initialise the optimizer
optimiser = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)

total_train_loss, total_val_loss, total_val_miou = [], [], []
for epoch in range(EPOCHS):

    # train for one epoch
    train_loss = train_epoch(model, train_loader, criterion, optimiser, epoch, device)

    # evaluate on validation set
    val_loss, miou = eval_epoch(model, val_loader, criterion, epoch, device)

    total_train_loss.append(train_loss); total_val_loss.append(val_loss); total_val_miou.append(miou)
    print(f"epoch {epoch+1:>2}/{EPOCHS}  train_loss={train_loss:.4f}  val_loss={val_loss:.4f}  val_mIoU={miou:.4f}")

# Plot training and validation curves
plt.plot(total_train_loss, label='Train')
plt.plot(total_val_loss, label='Val')
plt.legend()
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.show()

plt.plot(total_val_miou, label='Val')
plt.legend()
plt.xlabel('Epoch')
plt.ylabel('mIoU')
plt.show()

# 4. Evaluation

For segmentation, the first evaluation is always **visual**: a metric averaged over millions of pixels hides where the model actually fails. Let's colourise a few predictions from the validation set and compare them with the ground truth.

In [ ]:
# Visualise a few predictions from the validation set
CAMVID4_COLORS = np.array([[128,64,128],[0,0,142],[220,20,60],[70,70,70],[0,0,0]], np.uint8)
_MEAN = np.array([0.485, 0.456, 0.406]); _STD = np.array([0.229, 0.224, 0.225])

def denorm(t):
    """Undo the ImageNet normalisation, just for visualisation."""
    return np.clip(t.permute(1,2,0).numpy() * _STD + _MEAN, 0, 1)

def colourise(mask_np):
    m = mask_np.copy(); m[m == 255] = 4   # void => black
    return CAMVID4_COLORS[m]

model.eval()
imgs, masks = next(iter(val_loader))
with torch.no_grad():
    preds = model(imgs.to(device)).argmax(1).cpu()

n = min(4, len(imgs))
fig, axes = plt.subplots(n, 3, figsize=(12, 3*n))
for i in range(n):
    axes[i,0].imshow(denorm(imgs[i]));             axes[i,0].set_title("image");        axes[i,0].axis("off")
    axes[i,1].imshow(colourise(masks[i].numpy())); axes[i,1].set_title("ground truth"); axes[i,1].axis("off")
    axes[i,2].imshow(colourise(preds[i].numpy())); axes[i,2].set_title("prediction");   axes[i,2].axis("off")
plt.tight_layout(); plt.show()

Quantitatively, we report the IoU **per class** on the test set, together with the number of pixels each class covers. The two columns should be read together: rare classes (pedestrians) occupy a tiny fraction of the pixels and are much harder, which is exactly the **class imbalance** problem the Dice loss will address in the next section.

In [ ]:
# prediction on the test set, per-class IoU and number of pixels per class
test_loader = DataLoader(test_dataset, batch_size=BATCH, shuffle=False, num_workers=0, pin_memory=True)

model.eval()
avg_ious = []
avg_nb_pixels_class = []

for imgs, masks in tqdm.tqdm(test_loader, total=len(test_loader), desc='test phase'):
    imgs, masks = imgs.to(device), masks.to(device)
    with torch.no_grad():
        preds = model(imgs).argmax(1)

    ious = []
    nb_pixels_class = []
    for c in range(NUM_CLASSES):
        valid = masks != 255
        tp = ((preds == c) & (masks == c) & valid).sum().item()
        fp = ((preds == c) & (masks != c) & valid).sum().item()
        fn = ((preds != c) & (masks == c) & valid).sum().item()

        ious.append(tp / (tp + fp + fn) if tp + fp + fn > 0 else np.nan)
        nb_pixels_class.append((masks == c).sum().item())

    avg_ious.append(ious)
    avg_nb_pixels_class.append(nb_pixels_class)

In [ ]:
avg_ious_arr = np.array(avg_ious)                    # (num_batches, num_classes)
avg_nb_pixels_arr = np.array(avg_nb_pixels_class)

safe_ious_arr = np.nan_to_num(avg_ious_arr, nan=0.0) # batches where a class is absent do not count

total_pixels = avg_nb_pixels_arr.sum(axis=0)         # (num_classes,)
weighted_iou = (safe_ious_arr * avg_nb_pixels_arr).sum(axis=0) / total_pixels

class_names = ["Road", "Car", "Pedestrian", "Other"]
print(f"{'Class':<15} {'mIoU':>8} {'Avg Pixels':>12}")
print("-" * 37)
for i, name in enumerate(class_names):
    print(f"{name:<15} {weighted_iou[i]:>8.4f} {total_pixels[i]/len(avg_ious):>12.1f}")
print("-" * 37)
print(f"{'Mean':<15} {weighted_iou.mean():>8.4f}")

# 5. Binary Segmentation: the Crack Dataset

We now move to [DeepCrack](https://github.com/yhlleo/DeepCrack), a **binary** segmentation dataset: each pixel is either *crack* (1) or *background* (0). The layout is different from CamVid — there is no split file, images and labels simply live in parallel folders:

```
data/deepcrack/
    train_img/  train_lab/   300 pairs   (image .jpg, label .png)
    test_img/   test_lab/    237 pairs
```

Three things to watch:
* labels are stored as `{0, 255}` grey-scale PNGs, so we binarise them with `> 0`;
* cracks are **thin**: shrinking the image would erase them, so we train on random **256x256 crops** at native resolution rather than on downsampled frames;
* images come in **two orientations** (544x384 and 384x544). Crops sidestep this at training time; for validation and test we keep the full frame and rotate the portrait ones to landscape so they still stack into a batch.

In [ ]:
crack_root = "data/deepcrack"

first_image = f'{crack_root}/train_img/11111.jpg'
first_label = f'{crack_root}/train_lab/11111.png'

img = Image.open(first_image)
lbl = Image.open(first_label)

img_array = np.array(img)
lbl_array = np.array(lbl)

print(img_array.shape, lbl_array.shape)
print(np.unique(lbl_array))          # {0, 255} => binarise with > 0

plt.figure(figsize=(15, 5))
plt.subplot(1, 3, 1)
plt.imshow(img_array)
plt.title("Image")
plt.axis('off')
plt.subplot(1, 3, 2)
plt.imshow(lbl_array, cmap='gray')
plt.title("Label")
plt.axis('off')
plt.subplot(1, 3, 3)
plt.imshow(img_array)
plt.imshow(lbl_array, alpha=0.5, cmap='gray')
plt.title("Overlay")
plt.axis('off')
plt.show()

Same recipe as `CamVid`: list the pairs in `__init__`, and in `__getitem__` open / crop-or-resize / convert to tensor. Three differences — the pairs are built by matching filenames between `*_img` and `*_lab` (no split file), the remapping is a single `> 0` threshold instead of the 11 -> 4 class LUT, and a `crop` argument switches the dataset between **training mode** (random 256 crop, a new one every epoch, which also acts as data augmentation) and **evaluation mode** (full frame). There is no void class here, so nothing is ignored in the loss.

In [ ]:
W_crack, H_crack = 544, 384 # native landscape size; portrait images are rotated so full-res samples can be batched
CROP_crack = 256

normalise_crack = T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

class DeepCrack(Dataset):
    def __init__(self, root_dir, split, crop=None, transform=None):
        self.root_dir = root_dir
        self.crop = crop            # None => full resolution (validation / test)
        self.transform = transform
        names = sorted(os.listdir(f"{root_dir}/{split}_img"))   # e.g. "11111.jpg"

        self.image_paths = []
        self.label_paths = []
        for name in names:
            stem = os.path.splitext(name)[0]
            self.image_paths.append(f"{root_dir}/{split}_img/{stem}.jpg")
            self.label_paths.append(f"{root_dir}/{split}_lab/{stem}.png")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image_path = self.image_paths[idx]
        label_path = self.label_paths[idx]

        image = Image.open(image_path).convert('RGB')
        label = Image.open(label_path).convert('L')

        if self.crop:
            # Random crop at native resolution: no downsampling, so thin cracks survive
            i, j, h, w = T.RandomCrop.get_params(image, output_size=(self.crop, self.crop))
            image = TF.crop(image, i, j, h, w)
            label = TF.crop(label, i, j, h, w)
        else:
            # Mixed orientation: rotate the portrait images so every sample is landscape
            if image.height > image.width:
                image = image.transpose(Image.ROTATE_90)
                label = label.transpose(Image.ROTATE_90)
            image = TF.resize(image, (H_crack, W_crack))
            label = TF.resize(label, (H_crack, W_crack), interpolation=Image.NEAREST)

        # Binarise the {0, 255} mask => {0, 1}
        label_np = (np.array(label) > 0).astype(np.uint8)

        # Convert to tensor
        image = TF.to_tensor(image)
        label = torch.from_numpy(label_np).long()

        # Normalize
        if self.transform:
            image = self.transform(image)

        return image, label

In [ ]:
# Two views of the training folder: random crops to train on, full resolution to validate on
crops_train_dataset = DeepCrack(root_dir=crack_root, split="train", crop=CROP_crack, transform=normalise_crack)
full_train_dataset  = DeepCrack(root_dir=crack_root, split="train", transform=normalise_crack)
crack_test_dataset  = DeepCrack(root_dir=crack_root, split="test",  transform=normalise_crack)

# DeepCrack ships only train/test, so we carve a validation set out of the training split
n_val = int(0.2 * len(crops_train_dataset))
perm = torch.randperm(len(crops_train_dataset),
                      generator=torch.Generator().manual_seed(0)).tolist()   # fixed seed => reproducible split
crack_train_dataset = Subset(crops_train_dataset, perm[n_val:])
crack_val_dataset   = Subset(full_train_dataset,  perm[:n_val])

print(f"size of training dataset: {len(crack_train_dataset)}")
print(f"size of validation dataset: {len(crack_val_dataset)}")
print(f"size of test dataset: {len(crack_test_dataset)}")

## The Dice loss

Cracks cover roughly **2-3% of the pixels**. With a per-pixel cross-entropy, a network that predicts *background* everywhere already reaches ~97% pixel accuracy and a very low loss: the gradient signal from the crack pixels is drowned by the background ones.

The **Dice loss** measures the **overlap between two sets** instead of averaging per-pixel errors, so it is insensitive to how much background there is. Its soft (differentiable) form replaces the binary prediction by the predicted probability, which lets it be used as a training objective.

### Task 2: implement the binary dice loss

The soft dice coefficient between predicted foreground probabilities $p$ (sigmoid of the logits) and targets $g$ is

$$\text{Dice} = \frac{2\sum p g + \varepsilon}{\sum p + \sum g + \varepsilon},\qquad
\mathcal{L}_{\text{dice}} = 1 - \text{Dice}.$$

Complete the `forward` below:
1. Compute the **intersection** $\sum p g$ and the **denominator** $\sum p + \sum g$, summing over the batch **and** the spatial dimensions (one soft dice for the whole batch).
2. Combine them into the dice coefficient, with $\varepsilon$ smoothing on both sides to avoid a division by zero on a batch with no crack at all.
3. Return $1 - \text{Dice}$, so that a perfect overlap gives a loss of 0.

**Hint:** `p` and `g` are `(B, H, W)` tensors, so [`.sum()`](https://pytorch.org/docs/stable/generated/torch.sum.html) with no argument already reduces over every dimension. The head outputs a **single** channel here: for a binary problem one logit plus a [`sigmoid`](https://pytorch.org/docs/stable/generated/torch.sigmoid.html) is enough, and the prediction is `sigmoid(logit) > 0.5`.

In [ ]:
class DiceLoss(nn.Module):
    def __init__(self, epsilon=1e-6):
        super().__init__()
        self.epsilon = epsilon

    def forward(self, logits, target):
        logits = logits.float()                # compute in fp32
        p = torch.sigmoid(logits).squeeze(1)   # (B,H,W) foreground probability
        g = target.float()                     # (B,H,W) ground truth in {0, 1}

        # Sum over batch and spatial dims => one soft dice for the whole batch
        intersection = ...
        denominator = ...

        dice = ...
        return 1 - dice

## Atrous (dilated) convolution

Cracks are **thin**, elongated structures: a few pixels wide but spanning the whole image. A plain decoder sees them through a small receptive field, so it tends to break a crack into disconnected fragments. Downsampling further would widen the receptive field but destroy the very detail we are trying to segment.

**Atrous (dilated) convolution** solves this trade-off: a $3\times3$ kernel with dilation $d$ covers a $(2d+1)\times(2d+1)$ window with the same 9 weights, at the same cost, and **without losing resolution**. Stacking several dilation rates in parallel gives the **ASPP** (Atrous Spatial Pyramid Pooling) module of [DeepLab](https://arxiv.org/abs/1706.05587): each output pixel then sees local texture *and* long-range context at once.

The same idea is standard in medical imaging, where the targets are equally thin and elongated — retinal vessels, airways, wound boundaries — and where losing spatial resolution directly costs measurement accuracy.

### Task 3: implement the atrous module

Complete `ASPPLiteBlock` below. It is a *lite* ASPP: three parallel branches on the same input, fused into one output.

1. In `__init__`, create the **three parallel branches** with `conv_bn_relu` (defined in section 2, it already takes a `dilation` argument): `cin -> cout` with dilations **1**, **6** and **12**.
2. Create the **fusion** layer: the three branches are concatenated along the channel dimension, so it takes `cout * 3` channels and maps them back to `cout` with a $1\times1$ [`nn.Conv2d`](https://pytorch.org/docs/stable/generated/torch.nn.Conv2d.html), followed by `nn.BatchNorm2d` and `nn.ReLU`.
3. In `forward`, run the three branches on `x`, concatenate them with [`torch.cat`](https://pytorch.org/docs/stable/generated/torch.cat.html) along `dim=1`, and pass the result through the fusion layer.

**Hint:** all three branches must return the **same spatial size** as the input, otherwise the concatenation fails — this is why `conv_bn_relu` uses `padding=dilation`. `UNetLiteAtrous` (given) plugs the block on the `/16` encoder feature map, which has **256 channels**; everything else is identical to `UNetLite`, so any difference in the results comes from the ASPP block alone.

In [ ]:
# ASPP-lite: three parallel 3x3 convs with dilations 1 / 6 / 12, fused by a 1x1 conv
class ASPPLiteBlock(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        # three parallel branches, same kernel size, different dilations
        self.b1  = ...
        self.b6  = ...
        self.b12 = ...

        # 1x1 conv fusing the three concatenated branches back to cout channels
        self.fuse = ...

    def forward(self, x):
        # TODO: run the three branches, concatenate them on the channel dim, then fuse
        return ...


# same UNetLite as before, with an ASPP block on the /16 skip connection
class UNetLiteAtrous(nn.Module):
    def __init__(self, num_classes, freeze_encoder=True):
        super().__init__()
        self.encoder = ResNet18Encoder(freeze=freeze_encoder)
        self.aspp = ASPPLiteBlock(256, 256)   # /16 features have 256 channels

        self.dec3 = DecoderBlock(512, 256, 256)
        self.dec2 = DecoderBlock(256, 128, 128)
        self.dec1 = DecoderBlock(128,  64,  64)
        self.head = nn.Conv2d(64, num_classes, kernel_size=1)

    def forward(self, x):
        H, W = x.shape[-2:]
        c4, c8, c16, c32 = self.encoder(x)

        c16 = self.aspp(c16)   # widen the receptive field, keep the resolution
        x = self.dec3(c32, c16)
        x = self.dec2(x,   c8)
        x = self.dec1(x,   c4)
        x = self.head(x)
        return F.interpolate(x, size=(H, W), mode="bilinear", align_corners=False)

## Training both models

We now train the **same** network twice with the dice loss — `model_no_atrous` and `model_atrous` — starting from the same seed, so the ASPP block is the only difference between them.

Note the metrics: on a binary problem we report the **hard dice** and the **IoU** of the foreground (crack) class only. Background is never scored, otherwise every number would sit above 0.97 and tell us nothing.

In [ ]:
# any hyperparameters
EPOCHS_CRACK = 10
LR           = 5e-4
BATCH_CRACK  = 4

crack_train_loader = DataLoader(crack_train_dataset, batch_size=BATCH_CRACK, shuffle=True,  num_workers=0, pin_memory=True)
crack_val_loader   = DataLoader(crack_val_dataset,   batch_size=BATCH_CRACK, shuffle=False, num_workers=0, pin_memory=True)
crack_test_loader  = DataLoader(crack_test_dataset,  batch_size=BATCH_CRACK, shuffle=False, num_workers=0, pin_memory=True)

criterion = DiceLoss()


def binary_scores(preds, masks, epsilon=1e-6):
    """Hard dice and IoU of the crack (foreground) class."""
    tp = ((preds == 1) & (masks == 1)).sum().item()
    fp = ((preds == 1) & (masks == 0)).sum().item()
    fn = ((preds == 0) & (masks == 1)).sum().item()
    dice = (2 * tp + epsilon) / (2 * tp + fp + fn + epsilon)
    iou  = (tp + epsilon) / (tp + fp + fn + epsilon)
    return dice, iou


def train_binary(model, epochs, device):
    """Train one crack model with the dice loss, printing train / val stats per epoch."""
    model = model.to(device)
    optimiser = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)

    for epoch in range(epochs):
        # --- train ---
        model.train()
        train_loss = 0.0
        for imgs, masks in tqdm.tqdm(crack_train_loader, total=len(crack_train_loader), desc=f'Epoch {epoch+1} - training phase'):
            imgs, masks = imgs.to(device), masks.to(device)
            optimiser.zero_grad(set_to_none=True)
            loss = criterion(model(imgs), masks)
            loss.backward()
            optimiser.step()
            train_loss += loss.item()

        # --- validate ---
        model.eval()
        val_loss, all_preds, all_masks = 0.0, [], []
        with torch.no_grad():
            for imgs, masks in crack_val_loader:
                imgs, masks = imgs.to(device), masks.to(device)
                logits = model(imgs)
                val_loss += criterion(logits, masks).item()
                all_preds.append((logits.sigmoid() > 0.5).long().squeeze(1).cpu())
                all_masks.append(masks.cpu())

        dice, iou = binary_scores(torch.cat(all_preds), torch.cat(all_masks))
        print(f"epoch {epoch+1:>2}/{epochs}  "
              f"train_loss={train_loss/len(crack_train_loader):.4f}  "
              f"val_loss={val_loss/len(crack_val_loader):.4f}  "
              f"val_dice={dice:.4f}  val_IoU={iou:.4f}")

    return model

In [ ]:
# model 1: plain UNetLite, no atrous
torch.manual_seed(0)   # same init for both models => the only difference is the ASPP block

model_no_atrous = UNetLite(num_classes=1, freeze_encoder=True)
model_no_atrous = train_binary(model_no_atrous, EPOCHS_CRACK, device)

In [ ]:
# model 2: same network, with the ASPP-lite block on the /16 skip
torch.manual_seed(0)   # same seed as above => identical starting weights

model_atrous = UNetLiteAtrous(num_classes=1, freeze_encoder=True)
model_atrous = train_binary(model_atrous, EPOCHS_CRACK, device)

## Comparing the two models

Finally we score both models on the **test set** and look at their predictions side by side. Pay attention to **connectivity** rather than to the raw numbers: the interesting question is whether the atrous model keeps a crack as one continuous curve where the plain model breaks it into fragments.

In [ ]:
# prediction on the test set, both models
def evaluate_crack(model, loader):
    model.eval()
    all_preds, all_masks = [], []
    for imgs, masks in loader:
        imgs, masks = imgs.to(device), masks.to(device)
        with torch.no_grad():
            logits = model(imgs)
        all_preds.append((logits.sigmoid() > 0.5).long().squeeze(1).cpu())
        all_masks.append(masks.cpu())
    return binary_scores(torch.cat(all_preds), torch.cat(all_masks))


dice_no_atrous, iou_no_atrous = evaluate_crack(model_no_atrous, crack_test_loader)
dice_atrous,    iou_atrous    = evaluate_crack(model_atrous,    crack_test_loader)

print(f"{'model':<20} {'Dice':>8} {'IoU':>8}")
print("-" * 37)
print(f"{'no atrous':<20} {dice_no_atrous:>8.4f} {iou_no_atrous:>8.4f}")
print(f"{'atrous':<20} {dice_atrous:>8.4f} {iou_atrous:>8.4f}")
print("-" * 37)

In [ ]:
# Visualise a few predictions from the test set
n = 12
idxs = np.linspace(0, len(crack_test_dataset) - 1, n, dtype=int)   # spread over the whole test set

imgs  = torch.stack([crack_test_dataset[i][0] for i in idxs])
masks = torch.stack([crack_test_dataset[i][1] for i in idxs])

model_no_atrous.eval()
model_atrous.eval()
with torch.no_grad():
    preds_no_atrous = (model_no_atrous(imgs.to(device)).sigmoid() > 0.5).long().squeeze(1).cpu()
    preds_atrous    = (model_atrous(imgs.to(device)).sigmoid() > 0.5).long().squeeze(1).cpu()

fig, axes = plt.subplots(n, 4, figsize=(16, 3*n))
for i in range(n):
    axes[i,0].imshow(denorm(imgs[i]));                         axes[i,0].set_title("image");           axes[i,0].axis("off")
    axes[i,1].imshow(masks[i].numpy(), cmap="gray");           axes[i,1].set_title("ground truth");    axes[i,1].axis("off")
    axes[i,2].imshow(preds_no_atrous[i].numpy(), cmap="gray"); axes[i,2].set_title("dice, no atrous"); axes[i,2].axis("off")
    axes[i,3].imshow(preds_atrous[i].numpy(), cmap="gray");    axes[i,3].set_title("dice, atrous");    axes[i,3].axis("off")
plt.tight_layout(); plt.show()

## Going further

A few things worth experimenting with once the three tasks run:

1. **Unfreeze the encoder** (`freeze_encoder=False`) with a smaller learning rate. How much does the decoder alone limit us?
2. **Change the dilation rates** of the ASPP block (e.g. 1 / 2 / 4, or 1 / 12 / 24). Very large dilations on a small feature map degenerate into a $1\times1$ conv — at which rate does the block stop helping?
3. **Combine the losses**: `loss = dice + cross_entropy`. This is the usual recipe in medical imaging, where dice fixes the imbalance and cross-entropy keeps the gradients well behaved early in training.
4. **Move the ASPP block** to the `/32` feature map instead of `/16`, or add one at each scale, and compare the cost / accuracy trade-off.